# Neural Networks from Scratch

**Module 03: Deep Learning**  
**Objective**: Build neural networks from first principles using only NumPy

Understanding neural networks at the lowest level helps you:
- Debug gradient issues
- Implement custom architectures
- Optimize performance
- Answer deep technical interviews

## What You'll Learn
1. Forward propagation
2. Backpropagation (chain rule)
3. Activation functions and their gradients
4. Loss functions
5. Weight initialization strategies
6. Optimization algorithms (SGD, Momentum, Adam)
7. Regularization techniques
8. Building a complete training loop

## 1. Neuron: The Building Block

A single neuron computes:
$$z = w^Tx + b$$
$$a = \sigma(z)$$

Where:
- $w$: weights
- $x$: input
- $b$: bias
- $\sigma$: activation function
- $a$: output (activation)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_circles, make_classification
from sklearn.model_selection import train_test_split

np.random.seed(42)

In [ ]:
# Activation functions
def sigmoid(z):
    """Sigmoid activation."""
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def sigmoid_derivative(z):
    """Derivative of sigmoid."""
    s = sigmoid(z)
    return s * (1 - s)

def relu(z):
    """ReLU activation."""
    return np.maximum(0, z)

def relu_derivative(z):
    """Derivative of ReLU."""
    return (z > 0).astype(float)

def tanh(z):
    """Tanh activation."""
    return np.tanh(z)

def tanh_derivative(z):
    """Derivative of tanh."""
    return 1 - np.tanh(z) ** 2

def softmax(z):
    """Softmax activation (for multi-class)."""
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))  # Numerical stability
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

# Visualize activations
x = np.linspace(-5, 5, 100)

plt.figure(figsize=(14, 4))

plt.subplot(1, 3, 1)
plt.plot(x, sigmoid(x), label='sigmoid')
plt.plot(x, relu(x), label='ReLU')
plt.plot(x, tanh(x), label='tanh')
plt.xlabel('x')
plt.ylabel('activation')
plt.title('Activation Functions')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 2)
plt.plot(x, sigmoid_derivative(x), label='sigmoid\'')
plt.plot(x, relu_derivative(x), label='ReLU\'')
plt.plot(x, tanh_derivative(x), label='tanh\'')
plt.xlabel('x')
plt.ylabel('derivative')
plt.title('Activation Derivatives')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 3, 3)
# Softmax example
z = np.array([[1, 2, 3], [1, 1, 1], [3, 2, 1]])
probs = softmax(z)
for i, prob in enumerate(probs):
    plt.bar(range(3), prob, alpha=0.6, label=f'Sample {i+1}')
plt.xlabel('Class')
plt.ylabel('Probability')
plt.title('Softmax Output')
plt.legend()

plt.tight_layout()
plt.show()

print("Key properties:")
print("- Sigmoid: outputs (0, 1), smooth, but vanishing gradient problem")
print("- ReLU: outputs [0, ∞), sparse, no vanishing gradient, but dying ReLU problem")
print("- Tanh: outputs (-1, 1), zero-centered, still has vanishing gradient")
print("- Softmax: outputs probability distribution, used for multi-class classification")

## 2. Two-Layer Neural Network

**Architecture**:
```
Input (n_input) → Hidden (n_hidden) → Output (n_output)
```

**Forward pass**:
$$Z^{[1]} = W^{[1]}X + b^{[1]}$$
$$A^{[1]} = \sigma(Z^{[1]})$$
$$Z^{[2]} = W^{[2]}A^{[1]} + b^{[2]}$$
$$A^{[2]} = softmax(Z^{[2]})$$

In [ ]:
class TwoLayerNN:
    """Two-layer neural network for binary/multi-class classification."""
    
    def __init__(self, input_size, hidden_size, output_size, activation='relu'):
        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        
        # Initialize weights (He initialization for ReLU)
        self.W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2.0 / input_size)
        self.b1 = np.zeros((1, hidden_size))
        self.W2 = np.random.randn(hidden_size, output_size) * np.sqrt(2.0 / hidden_size)
        self.b2 = np.zeros((1, output_size))
        
        # Activation function
        if activation == 'relu':
            self.activation = relu
            self.activation_derivative = relu_derivative
        elif activation == 'sigmoid':
            self.activation = sigmoid
            self.activation_derivative = sigmoid_derivative
        elif activation == 'tanh':
            self.activation = tanh
            self.activation_derivative = tanh_derivative
        
        self.cache = {}
        self.losses = []
    
    def forward(self, X):
        """Forward propagation."""
        # Layer 1
        self.cache['Z1'] = X @ self.W1 + self.b1
        self.cache['A1'] = self.activation(self.cache['Z1'])
        
        # Layer 2
        self.cache['Z2'] = self.cache['A1'] @ self.W2 + self.b2
        
        # Output (softmax for multi-class, sigmoid for binary)
        if self.output_size > 1:
            self.cache['A2'] = softmax(self.cache['Z2'])
        else:
            self.cache['A2'] = sigmoid(self.cache['Z2'])
        
        return self.cache['A2']
    
    def compute_loss(self, y_true, y_pred):
        """Cross-entropy loss."""
        m = y_true.shape[0]
        
        if self.output_size > 1:
            # Multi-class cross-entropy
            log_probs = -np.log(y_pred[range(m), y_true] + 1e-8)
            loss = np.sum(log_probs) / m
        else:
            # Binary cross-entropy
            loss = -np.mean(y_true * np.log(y_pred + 1e-8) + 
                           (1 - y_true) * np.log(1 - y_pred + 1e-8))
        
        return loss
    
    def backward(self, X, y_true):
        """Backpropagation."""
        m = X.shape[0]
        
        # Output layer gradient
        if self.output_size > 1:
            # Multi-class: softmax + cross-entropy derivative
            dZ2 = self.cache['A2'].copy()
            dZ2[range(m), y_true] -= 1
            dZ2 /= m
        else:
            # Binary: sigmoid + binary cross-entropy
            dZ2 = (self.cache['A2'] - y_true) / m
        
        # Layer 2 gradients
        dW2 = self.cache['A1'].T @ dZ2
        db2 = np.sum(dZ2, axis=0, keepdims=True)
        
        # Hidden layer gradient
        dA1 = dZ2 @ self.W2.T
        dZ1 = dA1 * self.activation_derivative(self.cache['Z1'])
        
        # Layer 1 gradients
        dW1 = X.T @ dZ1
        db1 = np.sum(dZ1, axis=0, keepdims=True)
        
        return dW1, db1, dW2, db2
    
    def update_parameters(self, dW1, db1, dW2, db2, learning_rate):
        """Gradient descent update."""
        self.W1 -= learning_rate * dW1
        self.b1 -= learning_rate * db1
        self.W2 -= learning_rate * dW2
        self.b2 -= learning_rate * db2
    
    def train(self, X, y, epochs=1000, learning_rate=0.1, verbose=True):
        """Training loop."""
        for epoch in range(epochs):
            # Forward
            y_pred = self.forward(X)
            
            # Loss
            loss = self.compute_loss(y, y_pred)
            self.losses.append(loss)
            
            # Backward
            dW1, db1, dW2, db2 = self.backward(X, y)
            
            # Update
            self.update_parameters(dW1, db1, dW2, db2, learning_rate)
            
            # Log
            if verbose and epoch % 100 == 0:
                if self.output_size > 1:
                    acc = np.mean(y == np.argmax(y_pred, axis=1))
                else:
                    acc = np.mean(y == (y_pred > 0.5).astype(int))
                print(f"Epoch {epoch:4d}: Loss = {loss:.4f}, Accuracy = {acc:.4f}")
    
    def predict(self, X):
        """Make predictions."""
        y_pred = self.forward(X)
        if self.output_size > 1:
            return np.argmax(y_pred, axis=1)
        else:
            return (y_pred > 0.5).astype(int)

# Test on moons dataset
print("Testing Two-Layer Neural Network\n")
print("="*50)

X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train model
model = TwoLayerNN(input_size=2, hidden_size=16, output_size=2, activation='relu')
model.train(X_train, y_train, epochs=1000, learning_rate=0.5, verbose=True)

# Evaluate
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

train_acc = np.mean(y_train == y_pred_train)
test_acc = np.mean(y_test == y_pred_test)

print(f"\nFinal Results:")
print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
ax1.plot(model.losses)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.grid(True, alpha=0.3)

# Decision boundary
h = 0.01
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

ax2.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
ax2.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='viridis', edgecolors='k', s=40)
ax2.set_xlabel('Feature 1')
ax2.set_ylabel('Feature 2')
ax2.set_title(f'Decision Boundary (Test Acc = {test_acc:.3f})')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Deep Neural Network (L layers)

**Generalize to arbitrary depth**:
- Layer $\ell$: $n^{[\ell]}$ neurons
- Forward: $A^{[\ell]} = \sigma(W^{[\ell]}A^{[\ell-1]} + b^{[\ell]})$
- Backward: Chain rule through all layers

In [ ]:
class DeepNN:
    """Deep neural network with arbitrary layers."""
    
    def __init__(self, layer_dims, activations='relu'):
        """
        Args:
            layer_dims: list of layer sizes [input, hidden1, hidden2, ..., output]
            activations: 'relu' or 'tanh' for hidden layers
        """
        self.L = len(layer_dims) - 1  # Number of weight matrices
        self.layer_dims = layer_dims
        self.parameters = {}
        self.cache = {}
        self.losses = []
        
        # Initialize parameters
        for l in range(1, self.L + 1):
            self.parameters[f'W{l}'] = np.random.randn(layer_dims[l-1], layer_dims[l]) * np.sqrt(2.0 / layer_dims[l-1])
            self.parameters[f'b{l}'] = np.zeros((1, layer_dims[l]))
        
        self.activations = activations
    
    def forward(self, X):
        """Forward propagation through all layers."""
        self.cache['A0'] = X
        
        # Hidden layers
        for l in range(1, self.L):
            Z = self.cache[f'A{l-1}'] @ self.parameters[f'W{l}'] + self.parameters[f'b{l}']
            self.cache[f'Z{l}'] = Z
            
            if self.activations == 'relu':
                self.cache[f'A{l}'] = relu(Z)
            elif self.activations == 'tanh':
                self.cache[f'A{l}'] = tanh(Z)
        
        # Output layer
        Z_out = self.cache[f'A{self.L-1}'] @ self.parameters[f'W{self.L}'] + self.parameters[f'b{self.L}']
        self.cache[f'Z{self.L}'] = Z_out
        
        if self.layer_dims[-1] > 1:
            self.cache[f'A{self.L}'] = softmax(Z_out)
        else:
            self.cache[f'A{self.L}'] = sigmoid(Z_out)
        
        return self.cache[f'A{self.L}']
    
    def compute_loss(self, y_true, y_pred):
        """Cross-entropy loss."""
        m = y_true.shape[0]
        
        if self.layer_dims[-1] > 1:
            log_probs = -np.log(y_pred[range(m), y_true] + 1e-8)
            loss = np.sum(log_probs) / m
        else:
            loss = -np.mean(y_true * np.log(y_pred + 1e-8) + 
                           (1 - y_true) * np.log(1 - y_pred + 1e-8))
        
        return loss
    
    def backward(self, X, y_true):
        """Backpropagation through all layers."""
        m = X.shape[0]
        gradients = {}
        
        # Output layer gradient
        if self.layer_dims[-1] > 1:
            dZ = self.cache[f'A{self.L}'].copy()
            dZ[range(m), y_true] -= 1
            dZ /= m
        else:
            dZ = (self.cache[f'A{self.L}'] - y_true) / m
        
        # Backward through layers
        for l in reversed(range(1, self.L + 1)):
            gradients[f'dW{l}'] = self.cache[f'A{l-1}'].T @ dZ
            gradients[f'db{l}'] = np.sum(dZ, axis=0, keepdims=True)
            
            if l > 1:
                dA = dZ @ self.parameters[f'W{l}'].T
                if self.activations == 'relu':
                    dZ = dA * relu_derivative(self.cache[f'Z{l-1}'])
                elif self.activations == 'tanh':
                    dZ = dA * tanh_derivative(self.cache[f'Z{l-1}'])
        
        return gradients
    
    def update_parameters(self, gradients, learning_rate):
        """Gradient descent update."""
        for l in range(1, self.L + 1):
            self.parameters[f'W{l}'] -= learning_rate * gradients[f'dW{l}']
            self.parameters[f'b{l}'] -= learning_rate * gradients[f'db{l}']
    
    def train(self, X, y, epochs=1000, learning_rate=0.01, verbose=True):
        """Training loop."""
        for epoch in range(epochs):
            # Forward
            y_pred = self.forward(X)
            
            # Loss
            loss = self.compute_loss(y, y_pred)
            self.losses.append(loss)
            
            # Backward
            gradients = self.backward(X, y)
            
            # Update
            self.update_parameters(gradients, learning_rate)
            
            # Log
            if verbose and epoch % 100 == 0:
                if self.layer_dims[-1] > 1:
                    acc = np.mean(y == np.argmax(y_pred, axis=1))
                else:
                    acc = np.mean(y == (y_pred > 0.5).astype(int))
                print(f"Epoch {epoch:4d}: Loss = {loss:.4f}, Accuracy = {acc:.4f}")
    
    def predict(self, X):
        """Make predictions."""
        y_pred = self.forward(X)
        if self.layer_dims[-1] > 1:
            return np.argmax(y_pred, axis=1)
        else:
            return (y_pred > 0.5).astype(int)

# Test on circles dataset (nonlinear)
print("\n" + "="*50)
print("Testing Deep Neural Network\n")

X, y = make_circles(n_samples=1000, noise=0.1, factor=0.5, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Deep network: [2, 64, 32, 16, 2]
model = DeepNN(layer_dims=[2, 64, 32, 16,2], activations='relu')
model.train(X_train, y_train, epochs=1000, learning_rate=0.1, verbose=True)

# Evaluate
y_pred_test = model.predict(X_test)
test_acc = np.mean(y_test == y_pred_test)

print(f"\nDeep NN Test Accuracy: {test_acc:.4f}")

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
ax1.plot(model.losses)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss (Deep NN)')
ax1.grid(True, alpha=0.3)

# Decision boundary
h = 0.01
x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

ax2.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
ax2.scatter(X_test[:, 0], X_test[:, 1], c=y_test, cmap='viridis', edgecolors='k', s=40)
ax2.set_xlabel('Feature 1')
ax2.set_ylabel('Feature 2')
ax2.set_title(f'Deep NN Decision Boundary (Acc = {test_acc:.3f})')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Optimization Algorithms

**Problem**: Standard GD can be slow and get stuck in local minima.

**Solutions**:
1. **SGD**: Update on single sample (noisy but fast)
2. **Mini-batch GD**: Update on batch (balance)
3. **Momentum**: Accumulate gradients
4. **Adam**: Adaptive learning rates

In [ ]:
class NeuralNetworkWithOptimizers:
    """Neural network with different optimizers."""
    
    def __init__(self, layer_dims, optimizer='adam'):
        self.layer_dims = layer_dims
        self.L = len(layer_dims) - 1
        self.parameters = {}
        self.optimizer = optimizer
        
        # Initialize parameters
        for l in range(1, self.L + 1):
            self.parameters[f'W{l}'] = np.random.randn(layer_dims[l-1], layer_dims[l]) * 0.01
            self.parameters[f'b{l}'] = np.zeros((1, layer_dims[l]))
        
        # Optimizer-specific parameters
        if optimizer == 'momentum':
            self.velocity = {}
            for l in range(1, self.L + 1):
                self.velocity[f'VdW{l}'] = np.zeros_like(self.parameters[f'W{l}'])
                self.velocity[f'Vdb{l}'] = np.zeros_like(self.parameters[f'b{l}'])
        
        elif optimizer == 'adam':
            self.adam_m = {}  # First moment
            self.adam_v = {}  # Second moment
            self.t = 0  # Time step
            for l in range(1, self.L + 1):
                self.adam_m[f'mdW{l}'] = np.zeros_like(self.parameters[f'W{l}'])
                self.adam_m[f'mdb{l}'] = np.zeros_like(self.parameters[f'b{l}'])
                self.adam_v[f'vdW{l}'] = np.zeros_like(self.parameters[f'W{l}'])
                self.adam_v[f'vdb{l}'] = np.zeros_like(self.parameters[f'b{l}'])
        
        self.losses = []
    
    def forward(self, X):
        """Forward pass."""
        A = X
        self.cache = {'A0': X}
        
        for l in range(1, self.L + 1):
            Z = A @ self.parameters[f'W{l}'] + self.parameters[f'b{l}']
            A = relu(Z) if l < self.L else softmax(Z)
            self.cache[f'Z{l}'] = Z
            self.cache[f'A{l}'] = A
        
        return self.cache[f'A{self.L}']
    
    def compute_loss(self, y_true, y_pred):
        """Cross-entropy loss."""
        m = y_true.shape[0]
        log_probs = -np.log(y_pred[range(m), y_true] + 1e-8)
        return np.sum(log_probs) / m
    
    def backward(self, y_true):
        """Backward pass."""
        m = y_true.shape[0]
        gradients = {}
        
        dZ = self.cache[f'A{self.L}'].copy()
        dZ[range(m), y_true] -= 1
        dZ /= m
        
        for l in reversed(range(1, self.L + 1)):
            gradients[f'dW{l}'] = self.cache[f'A{l-1}'].T @ dZ
            gradients[f'db{l}'] = np.sum(dZ, axis=0, keepdims=True)
            
            if l > 1:
                dA = dZ @ self.parameters[f'W{l}'].T
                dZ = dA * relu_derivative(self.cache[f'Z{l-1}'])
        
        return gradients
    
    def update_parameters(self, gradients, learning_rate, beta=0.9, beta1=0.9, beta2=0.999, epsilon=1e-8):
        """Update parameters based on optimizer."""
        
        if self.optimizer == 'sgd':
            # Standard gradient descent
            for l in range(1, self.L + 1):
                self.parameters[f'W{l}'] -= learning_rate * gradients[f'dW{l}']
                self.parameters[f'b{l}'] -= learning_rate * gradients[f'db{l}']
        
        elif self.optimizer == 'momentum':
            # Momentum: v = beta*v + (1-beta)*grad
            for l in range(1, self.L + 1):
                self.velocity[f'VdW{l}'] = beta * self.velocity[f'VdW{l}'] + (1 - beta) * gradients[f'dW{l}']
                self.velocity[f'Vdb{l}'] = beta * self.velocity[f'Vdb{l}'] + (1 - beta) * gradients[f'db{l}']
                
                self.parameters[f'W{l}'] -= learning_rate * self.velocity[f'VdW{l}']
                self.parameters[f'b{l}'] -= learning_rate * self.velocity[f'Vdb{l}']
        
        elif self.optimizer == 'adam':
            # Adam: adaptive learning rates
            self.t += 1
            
            for l in range(1, self.L + 1):
                # Update first moment (mean)
                self.adam_m[f'mdW{l}'] = beta1 * self.adam_m[f'mdW{l}'] + (1 - beta1) * gradients[f'dW{l}']
                self.adam_m[f'mdb{l}'] = beta1 * self.adam_m[f'mdb{l}'] + (1 - beta1) * gradients[f'db{l}']
                
                # Update second moment (variance)
                self.adam_v[f'vdW{l}'] = beta2 * self.adam_v[f'vdW{l}'] + (1 - beta2) * (gradients[f'dW{l}'] ** 2)
                self.adam_v[f'vdb{l}'] = beta2 * self.adam_v[f'vdb{l}'] + (1 - beta2) * (gradients[f'db{l}'] ** 2)
                
                # Bias correction
                m_corrected_W = self.adam_m[f'mdW{l}'] / (1 - beta1 ** self.t)
                m_corrected_b = self.adam_m[f'mdb{l}'] / (1 - beta1 ** self.t)
                v_corrected_W = self.adam_v[f'vdW{l}'] / (1 - beta2 ** self.t)
                v_corrected_b = self.adam_v[f'vdb{l}'] / (1 - beta2 ** self.t)
                
                # Update parameters
                self.parameters[f'W{l}'] -= learning_rate * m_corrected_W / (np.sqrt(v_corrected_W) + epsilon)
                self.parameters[f'b{l}'] -= learning_rate * m_corrected_b / (np.sqrt(v_corrected_b) + epsilon)
    
    def train(self, X, y, epochs=1000, learning_rate=0.01, verbose=False):
        """Training loop."""
        for epoch in range(epochs):
            y_pred = self.forward(X)
            loss = self.compute_loss(y, y_pred)
            self.losses.append(loss)
            
            gradients = self.backward(y)
            self.update_parameters(gradients, learning_rate)
            
            if verbose and epoch % 100 == 0:
                acc = np.mean(y == np.argmax(y_pred, axis=1))
                print(f"Epoch {epoch}: Loss = {loss:.4f}, Acc = {acc:.4f}")
    
    def predict(self, X):
        """Predictions."""
        y_pred = self.forward(X)
        return np.argmax(y_pred, axis=1)

# Compare optimizers
print("\n" + "="*50)
print("Comparing Optimizers\n")

X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)

optimizers = ['sgd', 'momentum', 'adam']
results = {}

for opt in optimizers:
    print(f"Training with {opt.upper()}...")
    model = NeuralNetworkWithOptimizers(layer_dims=[2, 32, 16, 2], optimizer=opt)
    model.train(X, y, epochs=500, learning_rate=0.01, verbose=False)
    results[opt] = model.losses

# Plot comparison
plt.figure(figsize=(10, 6))
for opt, losses in results.items():
    plt.plot(losses, label=opt.upper())
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Optimizer Comparison')
plt.legend()
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.show()

print("\nKey takeaways:")
print("- SGD: Simple, but can be slow")
print("- Momentum: Smoother convergence, escapes shallow local minima")
print("- Adam: Fast convergence, adaptive learning rates, most widely used")

## Summary

You've built neural networks from scratch:
- ✅ Forward propagation (matrix multiplications)
- ✅ Backpropagation (chain rule derivatives)
- ✅ Activation functions (ReLU, Sigmoid, Tanh, Softmax)
- ✅ Loss functions (Cross-entropy)
- ✅ Deep networks (arbitrary layers)
- ✅ Optimization algorithms (SGD, Momentum, Adam)

**Key insights**:
1. Backpropagation is just calculus (chain rule)
2. ReLU is preferred over sigmoid (no vanishing gradient)
3. Adam optimizer works well most of the time
4. Proper weight initialization prevents gradient issues

**Next Notebook**: PyTorch implementation with batch normalization, dropout, and modern techniques!